<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/set_date.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)

print("folder ready")

Mounted at /content/drive
folder ready


In [ ]:
%%writefile /content/drive/MyDrive/ml_project/set_date.py
# -*- coding: utf-8 -*-
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

TARGET_DIR = "/content/drive/MyDrive/ml_project"
INPUT_PICKLE = os.path.join(TARGET_DIR, "df_selected.pkl")
CHOICE_FILE = os.path.join(TARGET_DIR, "date_settings.txt")
OUTPUT_PICKLE = os.path.join(TARGET_DIR, "df_date.pkl")

# ---------------------------------------------------------
# set_date_ui
# automatically loads df_selected.pkl from drive
# user chooses date column + fill method
# ---------------------------------------------------------
def set_date_ui():

    if not os.path.exists(INPUT_PICKLE):
        raise FileNotFoundError("df_selected.pkl not found on drive")

    data = pd.read_pickle(INPUT_PICKLE)

    label_col = widgets.Label("select the date column:")
    date_dropdown = widgets.Dropdown(options=data.columns.tolist())

    label_fill = widgets.Label("select missing fill method:")
    fill_dropdown = widgets.Dropdown(options=["zero", "ffill", "mean"])

    btn = widgets.Button(description="confirm")
    out = widgets.Output()

    def on_click(b):
        out.clear_output()

        date_col = date_dropdown.value
        fill_method = fill_dropdown.value

        with open(CHOICE_FILE, "w") as f:
            f.write(date_col + "\n")
            f.write(fill_method + "\n")

        with out:
            print("saved:", CHOICE_FILE)
            print("date column:", date_col)
            print("fill method:", fill_method)

    btn.on_click(on_click)

    display(widgets.VBox([
        label_col, date_dropdown,
        label_fill, fill_dropdown,
        btn, out
    ]))


# ---------------------------------------------------------
# set_date
# loads df_selected.pkl + date_settings.txt
# applies datetime conversion and missing fill
# saves df_date.pkl
# ---------------------------------------------------------
def set_date():

    if not os.path.exists(INPUT_PICKLE):
        raise FileNotFoundError("df_selected.pkl not found")

    if not os.path.exists(CHOICE_FILE):
        raise FileNotFoundError("date_settings.txt not found")

    data = pd.read_pickle(INPUT_PICKLE)

    with open(CHOICE_FILE, "r") as f:
        lines = f.readlines()
        date_col = lines[0].strip()
        fill = lines[1].strip()

    data[date_col] = pd.to_datetime(data[date_col], errors="coerce")
    data = data.sort_values(by=date_col)

    if fill == "zero":
        data[date_col] = data[date_col].fillna(pd.Timestamp(0))
    elif fill == "ffill":
        data[date_col] = data[date_col].fillna(method="ffill")
    elif fill == "mean":
        mean_val = data[date_col].dropna().mean()
        data[date_col] = data[date_col].fillna(mean_val)

    data.to_pickle(OUTPUT_PICKLE)
    print("saved:", OUTPUT_PICKLE)

    return data


Overwriting /content/drive/MyDrive/ml_project/set_date.py


In [ ]:
#NOWY KOD



%%writefile /content/drive/MyDrive/ml_project/set_date.py
# -*- coding: utf-8 -*-

import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

TARGET_DIR = "/content/drive/MyDrive/ml_project"
INPUT_PICKLE = os.path.join(TARGET_DIR, "df_selected.pkl")
CHOICE_FILE = os.path.join(TARGET_DIR, "date_settings.txt")
OUTPUT_PICKLE = os.path.join(TARGET_DIR, "df_date.pkl")

# ---------------------------------------------------------
# set_date_ui
# user chooses date column + fill method
# ---------------------------------------------------------
def set_date_ui():

    if not os.path.exists(INPUT_PICKLE):
        raise FileNotFoundError("df_selected.pkl not found on drive")

    data = pd.read_pickle(INPUT_PICKLE)

    label_col = widgets.Label("select the date column:")
    date_dropdown = widgets.Dropdown(options=data.columns.tolist())

    label_fill = widgets.Label("select missing fill method:")
    fill_dropdown = widgets.Dropdown(options=["zero", "ffill", "mean"])

    btn = widgets.Button(description="confirm")
    out = widgets.Output()

    def on_click(b):
        out.clear_output()

        date_col = date_dropdown.value
        fill_method = fill_dropdown.value

        with open(CHOICE_FILE, "w") as f:
            f.write(date_col + "\n")
            f.write(fill_method + "\n")

        with out:
            print("saved:", CHOICE_FILE)
            print("date column:", date_col)
            print("fill method:", fill_method)

    btn.on_click(on_click)

    display(widgets.VBox([
        label_col, date_dropdown,
        label_fill, fill_dropdown,
        btn, out
    ]))


# ---------------------------------------------------------
# set_date
# validates and normalizes time axis
# ---------------------------------------------------------
def set_date():

    if not os.path.exists(INPUT_PICKLE):
        raise FileNotFoundError("df_selected.pkl not found")

    if not os.path.exists(CHOICE_FILE):
        raise FileNotFoundError("date_settings.txt not found")

    data = pd.read_pickle(INPUT_PICKLE)

    with open(CHOICE_FILE, "r") as f:
        lines = f.readlines()
        date_col = lines[0].strip()
        fill = lines[1].strip()

    # 1. convert to datetime (format validation only)
    data[date_col] = pd.to_datetime(data[date_col], errors="coerce")

    # remove rows with invalid dates
    data = data.dropna(subset=[date_col])

    # 2. aggregate duplicate dates (mean for numeric columns)
    data = (
        data
        .groupby(date_col, as_index=False)
        .mean(numeric_only=True)
    )

    # 3. build full daily date range
    start_date = data[date_col].min()
    end_date = data[date_col].max()
    full_range = pd.date_range(start=start_date, end=end_date, freq="D")

    # 4. reindex to enforce continuous time axis
    data = (
        data
        .set_index(date_col)
        .reindex(full_range)
    )

    data.index.name = date_col

    # 5. fill missing values according to chosen method
    if fill == "zero":
        data = data.fillna(0)

    elif fill == "ffill":
        data = data.ffill()

    elif fill == "mean":
        data = data.fillna(data.mean(numeric_only=True))

    # 6. final sorting and save
    data = data.sort_index()
    data.to_pickle(OUTPUT_PICKLE)

    print("saved:", OUTPUT_PICKLE)

    return data


In [2]:
#NEXT NEW CODE


%%writefile /content/drive/MyDrive/ml_project/set_date.py
# -*- coding: utf-8 -*-

import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

TARGET_DIR = "/content/drive/MyDrive/ml_project"
INPUT_PICKLE = os.path.join(TARGET_DIR, "df_selected.pkl")
CHOICE_FILE = os.path.join(TARGET_DIR, "date_settings.txt")
OUTPUT_PICKLE = os.path.join(TARGET_DIR, "df_date.pkl")

# ---------------------------------------------------------
# set_date_ui
# user chooses:
# - date column
# - duplicate handling method
# - missing fill method
# ---------------------------------------------------------
def set_date_ui():

    if not os.path.exists(INPUT_PICKLE):
        raise FileNotFoundError("df_selected.pkl not found on drive")

    data = pd.read_pickle(INPUT_PICKLE)

    label_col = widgets.Label("select the date column:")
    date_dropdown = widgets.Dropdown(options=data.columns.tolist())

    label_dup = widgets.Label("select duplicate handling method:")
    dup_dropdown = widgets.Dropdown(
        options=["mean", "sum", "first", "last"]
    )

    label_fill = widgets.Label("select missing fill method:")
    fill_dropdown = widgets.Dropdown(
        options=["zero", "ffill", "mean"]
    )

    btn = widgets.Button(description="confirm")
    out = widgets.Output()

    def on_click(b):
        out.clear_output()

        date_col = date_dropdown.value
        dup_method = dup_dropdown.value
        fill_method = fill_dropdown.value

        with open(CHOICE_FILE, "w") as f:
            f.write(date_col + "\n")
            f.write(dup_method + "\n")
            f.write(fill_method + "\n")

        with out:
            print("saved:", CHOICE_FILE)
            print("date column:", date_col)
            print("duplicate method:", dup_method)
            print("fill method:", fill_method)

    btn.on_click(on_click)

    display(widgets.VBox([
        label_col, date_dropdown,
        label_dup, dup_dropdown,
        label_fill, fill_dropdown,
        btn, out
    ]))


# ---------------------------------------------------------
# set_date
# validates and normalizes time axis
# ---------------------------------------------------------
def set_date():

    if not os.path.exists(INPUT_PICKLE):
        raise FileNotFoundError("df_selected.pkl not found")

    if not os.path.exists(CHOICE_FILE):
        raise FileNotFoundError("date_settings.txt not found")

    data = pd.read_pickle(INPUT_PICKLE)

    with open(CHOICE_FILE, "r") as f:
        lines = f.readlines()
        date_col = lines[0].strip()
        dup_method = lines[1].strip()
        fill = lines[2].strip()

    # 1. convert to datetime
    data[date_col] = pd.to_datetime(data[date_col], errors="coerce")
    data = data.dropna(subset=[date_col])

    # 2. handle duplicate dates (USER CHOICE)
    if dup_method == "mean":
        data = data.groupby(date_col, as_index=False).mean(numeric_only=True)

    elif dup_method == "sum":
        data = data.groupby(date_col, as_index=False).sum(numeric_only=True)

    elif dup_method == "first":
        data = data.groupby(date_col, as_index=False).first()

    elif dup_method == "last":
        data = data.groupby(date_col, as_index=False).last()

    else:
        raise ValueError("unknown duplicate handling method")

    # 3. build full daily date range
    start_date = data[date_col].min()
    end_date = data[date_col].max()
    full_range = pd.date_range(start=start_date, end=end_date, freq="D")

    # 4. reindex to enforce continuous time axis
    data = data.set_index(date_col).reindex(full_range)
    data.index.name = date_col

    # 5. fill missing values (USER CHOICE)
    if fill == "zero":
        data = data.fillna(0)

    elif fill == "ffill":
        data = data.ffill()

    elif fill == "mean":
        data = data.fillna(data.mean(numeric_only=True))

    else:
        raise ValueError("unknown fill method")

    # 6. final sort and save
    data = data.sort_index()
    data.to_pickle(OUTPUT_PICKLE)

    print("saved:", OUTPUT_PICKLE)

    return data


Overwriting /content/drive/MyDrive/ml_project/set_date.py


In [5]:
#NEXT NEWEST CODE!!

%%writefile /content/drive/MyDrive/ml_project/set_date.py
# -*- coding: utf-8 -*-

import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

TARGET_DIR = "/content/drive/MyDrive/ml_project"
INPUT_PICKLE = os.path.join(TARGET_DIR, "df_selected.pkl")
CHOICE_FILE = os.path.join(TARGET_DIR, "date_settings.txt")
OUTPUT_PICKLE = os.path.join(TARGET_DIR, "df_date.pkl")

# ---------------------------------------------------------
# set_date_ui
# user chooses:
# - date column
# - duplicate handling method (numeric columns)
# - missing fill method
# ---------------------------------------------------------
def set_date_ui():

    if not os.path.exists(INPUT_PICKLE):
        raise FileNotFoundError("df_selected.pkl not found on drive")

    data = pd.read_pickle(INPUT_PICKLE)

    label_col = widgets.Label("select the date column:")
    date_dropdown = widgets.Dropdown(options=data.columns.tolist())

    label_dup = widgets.Label("select duplicate handling method (numeric):")
    dup_dropdown = widgets.Dropdown(
        options=["mean", "sum"]
    )

    label_fill = widgets.Label("select missing fill method:")
    fill_dropdown = widgets.Dropdown(
        options=["zero", "ffill", "mean"]
    )

    # >>> NEW: information about text columns <<<
    info_text = widgets.HTML(
        "<i>note: non-numeric (text) columns are always preserved and "
        "aggregated using the first value per day.</i>"
    )

    btn = widgets.Button(description="confirm")
    out = widgets.Output()

    def on_click(b):
        out.clear_output()

        date_col = date_dropdown.value
        dup_method = dup_dropdown.value
        fill_method = fill_dropdown.value

        with open(CHOICE_FILE, "w") as f:
            f.write(date_col + "\n")
            f.write(dup_method + "\n")
            f.write(fill_method + "\n")

        with out:
            print("saved:", CHOICE_FILE)
            print("date column:", date_col)
            print("duplicate method (numeric):", dup_method)
            print("fill method:", fill_method)

    btn.on_click(on_click)

    display(widgets.VBox([
        label_col, date_dropdown,
        label_dup, dup_dropdown,
        label_fill, fill_dropdown,
        info_text,
        btn, out
    ]))


# ---------------------------------------------------------
# set_date
# validates and normalizes time axis
# ---------------------------------------------------------
def set_date():

    if not os.path.exists(INPUT_PICKLE):
        raise FileNotFoundError("df_selected.pkl not found")

    if not os.path.exists(CHOICE_FILE):
        raise FileNotFoundError("date_settings.txt not found")

    data = pd.read_pickle(INPUT_PICKLE)

    with open(CHOICE_FILE, "r") as f:
        lines = f.readlines()
        date_col = lines[0].strip()
        dup_method = lines[1].strip()
        fill = lines[2].strip()

    # 1. convert to datetime and drop invalid dates
    data[date_col] = pd.to_datetime(data[date_col], errors="coerce")
    data = data.dropna(subset=[date_col])

    # 2. split columns by type
    num_cols = data.select_dtypes(include="number").columns.tolist()
    txt_cols = data.select_dtypes(exclude="number").columns.tolist()
    txt_cols = [c for c in txt_cols if c != date_col]

    # 3. aggregate duplicates
    if dup_method == "mean":
        num_agg = data.groupby(date_col)[num_cols].mean()
    elif dup_method == "sum":
        num_agg = data.groupby(date_col)[num_cols].sum()
    else:
        raise ValueError("unknown duplicate handling method")

    # text columns: always keep first value of the day
    if txt_cols:
        txt_agg = data.groupby(date_col)[txt_cols].first()
        data = pd.concat([num_agg, txt_agg], axis=1).reset_index()
    else:
        data = num_agg.reset_index()

    # 4. build full daily date range
    start_date = data[date_col].min()
    end_date = data[date_col].max()
    full_range = pd.date_range(start=start_date, end=end_date, freq="D")

    # 5. reindex to enforce continuous time axis
    data = data.set_index(date_col).reindex(full_range)
    data.index.name = date_col

    # 6. fill missing values (USER CHOICE)
    if fill == "zero":
        data[num_cols] = data[num_cols].fillna(0)

    elif fill == "ffill":
        data = data.ffill()

    elif fill == "mean":
        data[num_cols] = data[num_cols].fillna(data[num_cols].mean())

    else:
        raise ValueError("unknown fill method")

    # 7. final sort and save
    data = data.sort_index()
    data.to_pickle(OUTPUT_PICKLE)

    print("saved:", OUTPUT_PICKLE)

    return data



Overwriting /content/drive/MyDrive/ml_project/set_date.py
